# FUNGI-TUBE: Fixed Time-Window (FTW) Featurization Pipeline

**Purpose:** Extract a deterministic, fixed-window feature set from curated FUNGI-TUBE time-series data.

Unlike segmented regression (which fits piecewise-linear models with data-driven breakpoints),
this pipeline uses **fixed temporal boundaries** to partition each run into early, mid, and late
phases. This produces stable, reproducible features regardless of signal shape.

### Feature Architecture (36 features per signal, 12 signals = 432 total)

| Group | Count | Description |
|---|---|---|
| Global statistics | 8 | mean, std, range, final value, delta, skewness, kurtosis, AUC |
| Rate features | 6 | max absolute rate, time of max rate, max pos/neg rates, smoothed variants |
| Window statistics | 15 | mean, std, range, AUC, mean rate × 3 windows (early/mid/late) |
| Window ratios | 2 | mid-to-early mean ratio, late-to-mid mean ratio |
| Threshold timing | 5 | hours to reach 10%, 25%, 50%, 75%, 90% of total signal range |

### Time Windows
- **Early:** 0–96 hours (first ~4 days — lag / adaptation phase)
- **Mid:** 96–216 hours (~days 4–9 — active growth phase)
- **Late:** 216+ hours (~day 9 onward — maturation / stationary phase)

### Signals Extracted (12 families)
`cap_dt`, `cap_zs`, `co2_rel`, `voc_raw`, `voc_rh`, `ir_delta`,
`grayscale`, `mbi`, `co2_cum`, `h2o_cum`, `re`, `hagr`

## 1. Configuration & Setup

Set `DATA_DIR`, `OUTPUT_FILENAME`, and `FILE_PATTERN` before running. This notebook reads curated feature CSVs from `DATA_DIR` and writes one compiled fixed-window feature table back to that directory.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# Configuration

# Directory containing curated_features_*.csv files
DATA_DIR = os.path.dirname(os.path.abspath('__file__'))

# Output filename for compiled features
OUTPUT_FILENAME = 'ftw_features_ALL_compiled.csv'

# File pattern for curated time-series data
FILE_PATTERN = 'curated_features_*_datalog.csv'

# --- Time Window Boundaries (hours) ---
# These define the early/mid/late phase boundaries.
# Adjust based on your organism's growth kinetics.
WINDOWS = {
    'early': (0, 96),      # 0–4 days: lag / adaptation
    'mid':   (96, 216),    # 4–9 days: active growth
    'late':  (216, 9999),  # 9+ days:  maturation / stationary
}

# --- Threshold Levels for Timing Features ---
# Fraction of total signal range at which to record crossing time
THRESHOLDS = [0.10, 0.25, 0.50, 0.75, 0.90]

# --- Rate Smoothing Window ---
# Number of samples for rolling mean when computing smoothed max rate.
# At 10-min intervals, 6 samples = 1 hour smoothing window.
RATE_SMOOTH_WINDOW = 6

print("Configuration loaded.")
print(f"  Data directory:    {DATA_DIR}")
print(f"  File pattern:      {FILE_PATTERN}")
print(f"  Output file:       {OUTPUT_FILENAME}")
print(f"  Windows:           {WINDOWS}")
print(f"  Thresholds:        {THRESHOLDS}")
print(f"  Rate smooth window: {RATE_SMOOTH_WINDOW} samples")

In [ ]:
# Signal mapping
# Maps short feature-name prefixes to column names in the curated data.
# Each entry: 'output_prefix' -> 'curated_column_name'
#
# To add a new signal:
#   1. Add it to this dictionary
#   2. Ensure the column exists in your curated_features files
#   3. Re-run the notebook

SIGNALS = {
    # --- Capacitance (dielectric proxy for biomass/moisture) ---
    'cap_dt':    'Capacitance_detrended',              # Temperature-detrended capacitance
    'cap_zs':    'CapacitanceRaw_zscore',               # Z-scored raw capacitance

    # --- Gas Phase ---
    'co2_rel':   'CO2ppm_relChange_tempDetrended',      # Relative CO2 change (temp-corrected)
    'voc_raw':   'VOC_Raw_relChange',                   # Volatile organic compounds (raw)
    'voc_rh':    'VOC_RH_relChange',                    # VOC corrected for humidity

    # --- Thermal ---
    'ir_delta':  'IR_Delta_relChange',                  # IR temperature differential

    # --- Optical ---
    'grayscale': 'Grayscale',                           # RGB grayscale (substrate darkening)

    # --- Derived / Composite ---
    'mbi':       'MetabolicBalanceIndex',               # Metabolic balance index
    'co2_cum':   'Total_CO2_mg',                        # Cumulative CO2 production (mg)
    'h2o_cum':   'Total_H2O_mg',                        # Cumulative H2O loss (mg)
    're':        'RespirationEfficiency_ratio_zscore',   # Respiration efficiency ratio
    'hagr':      'HeatAdjustedGrowth_ratio_zscore',     # Heat-adjusted growth ratio
}

print(f"Signal mapping defined: {len(SIGNALS)} signals")
print(f"Expected output: {len(SIGNALS) * 36} features per run")
print()
for prefix, col in SIGNALS.items():
    print(f"  {prefix:<12s} <- {col}")

## 2. Feature Extraction Functions

Each function below computes one category of features from a time-series signal.
All functions accept a `hours` array and `values` array of equal length and return
a dictionary of `{feature_name: value}` pairs.

In [ ]:
# Global statistics
# Summary statistics computed over the entire run duration.
# These capture the overall magnitude, spread, and shape of
# each signal without any temporal windowing.

def compute_global_stats(prefix, hours, values):
    """
    Compute 8 global summary statistics for a signal.

    Parameters
    ----------
    prefix : str
        Feature name prefix (e.g., 'cap_dt')
    hours : np.ndarray
        Time axis in hours
    values : np.ndarray
        Signal values (same length as hours)

    Returns
    -------
    dict : {feature_name: value}
        Keys: mean, std, range, final, final_minus_initial,
              skewness, kurtosis, auc_total
    """
    feats = {}
    v = values[np.isfinite(values)]

    if len(v) == 0:
        # Return NaN for all features if signal is entirely missing
        for suffix in ['mean', 'std', 'range', 'final', 'final_minus_initial',
                       'skewness', 'kurtosis', 'auc_total']:
            feats[f'{prefix}_{suffix}'] = np.nan
        return feats

    feats[f'{prefix}_mean'] = np.mean(v)
    feats[f'{prefix}_std'] = np.std(v, ddof=1) if len(v) > 1 else 0.0
    feats[f'{prefix}_range'] = np.ptp(v)  # max - min
    feats[f'{prefix}_final'] = v[-1]
    feats[f'{prefix}_final_minus_initial'] = v[-1] - v[0]

    # Skewness (Fisher's definition, bias-corrected)
    # Measures asymmetry: positive = right-tailed, negative = left-tailed
    if len(v) > 2:
        m = np.mean(v)
        s = np.std(v, ddof=1)
        if s > 1e-12:
            n = len(v)
            skew = (n / ((n-1)*(n-2))) * np.sum(((v - m) / s) ** 3)
        else:
            skew = 0.0
    else:
        skew = 0.0
    feats[f'{prefix}_skewness'] = skew

    # Kurtosis (excess kurtosis; normal distribution = 0)
    # Measures tail heaviness: positive = heavy tails, negative = light tails
    if len(v) > 3:
        m = np.mean(v)
        s = np.std(v, ddof=1)
        if s > 1e-12:
            n = len(v)
            kurt = ((n*(n+1)) / ((n-1)*(n-2)*(n-3))) * np.sum(((v - m)/s)**4) \
                   - (3*(n-1)**2) / ((n-2)*(n-3))
        else:
            kurt = 0.0
    else:
        kurt = 0.0
    feats[f'{prefix}_kurtosis'] = kurt

    # AUC via trapezoidal integration (total area under the curve)
    # Uses actual hour values for proper time-weighting
    valid_mask = np.isfinite(values)
    h_valid = hours[valid_mask]
    v_valid = values[valid_mask]
    if len(v_valid) > 1:
        feats[f'{prefix}_auc_total'] = np.trapz(v_valid, h_valid)
    else:
        feats[f'{prefix}_auc_total'] = 0.0

    return feats

print("Global statistics function defined (8 features per signal).")

In [ ]:
# Rate features
# Capture the dynamics of signal change: how fast is the signal
# changing, when does the fastest change occur, and what does
# the rate look like after smoothing out measurement noise?

def compute_rate_features(prefix, hours, values, smooth_window=6):
    """
    Compute 6 rate-based features from a signal.

    Parameters
    ----------
    prefix : str
        Feature name prefix
    hours : np.ndarray
        Time axis in hours
    values : np.ndarray
        Signal values
    smooth_window : int
        Rolling window size for smoothed rate (default: 6 = 1 hour at 10-min intervals)

    Returns
    -------
    dict : {feature_name: value}
        Keys: max_abs_rate, max_rate_time_h, max_pos_rate,
              max_neg_rate, smooth_max_rate, smooth_max_rate_time_h
    """
    feats = {}

    # Compute point-to-point rate of change (delta_value / delta_time)
    if len(values) < 2:
        for suffix in ['max_abs_rate', 'max_rate_time_h', 'max_pos_rate',
                       'max_neg_rate', 'smooth_max_rate', 'smooth_max_rate_time_h']:
            feats[f'{prefix}_{suffix}'] = np.nan
        return feats

    dv = np.diff(values)
    dh = np.diff(hours)
    dh[dh == 0] = 1e-6  # Prevent division by zero
    rates = dv / dh

    # Handle NaN rates
    valid_rates = np.where(np.isfinite(rates), rates, 0.0)

    # Max absolute rate and its timing
    abs_rates = np.abs(valid_rates)
    idx_max = np.argmax(abs_rates)
    feats[f'{prefix}_max_abs_rate'] = abs_rates[idx_max]
    feats[f'{prefix}_max_rate_time_h'] = hours[idx_max + 1]  # +1 because diff shifts index

    # Max positive rate (fastest increase)
    pos_rates = valid_rates.copy()
    pos_rates[pos_rates < 0] = 0
    feats[f'{prefix}_max_pos_rate'] = np.max(pos_rates)

    # Max negative rate (fastest decrease, stored as positive magnitude)
    neg_rates = valid_rates.copy()
    neg_rates[neg_rates > 0] = 0
    feats[f'{prefix}_max_neg_rate'] = np.abs(np.min(neg_rates))

    # Smoothed rate: apply rolling mean before finding max
    # This reduces the influence of single-point noise spikes
    if len(valid_rates) >= smooth_window:
        kernel = np.ones(smooth_window) / smooth_window
        smooth_rates = np.convolve(np.abs(valid_rates), kernel, mode='valid')
        idx_smooth_max = np.argmax(smooth_rates)
        feats[f'{prefix}_smooth_max_rate'] = smooth_rates[idx_smooth_max]
        # Map smoothed index back to original time axis
        offset = smooth_window // 2
        feats[f'{prefix}_smooth_max_rate_time_h'] = hours[min(idx_smooth_max + offset, len(hours)-1)]
    else:
        feats[f'{prefix}_smooth_max_rate'] = feats[f'{prefix}_max_abs_rate']
        feats[f'{prefix}_smooth_max_rate_time_h'] = feats[f'{prefix}_max_rate_time_h']

    return feats

print("Rate features function defined (6 features per signal).")

In [ ]:
# Windowed statistics
# Split the time-series into early, mid, and late windows and
# compute summary statistics within each. This captures how the
# signal behaves during distinct growth phases without requiring
# breakpoint detection.
#
# Also computes two window ratios that capture the relative
# magnitude shift between adjacent phases.

def compute_window_features(prefix, hours, values, windows):
    """
    Compute 17 windowed features (5 stats × 3 windows + 2 ratios).

    Parameters
    ----------
    prefix : str
        Feature name prefix
    hours : np.ndarray
        Time axis in hours
    values : np.ndarray
        Signal values
    windows : dict
        Window definitions: {'early': (0, 96), 'mid': (96, 216), 'late': (216, 9999)}

    Returns
    -------
    dict : {feature_name: value}
        Per window: mean, std, range, auc, mean_rate
        Ratios: mid_early_ratio, late_mid_ratio
    """
    feats = {}
    window_means = {}

    for win_name, (t_start, t_end) in windows.items():
        # Select data points within the time window
        mask = (hours >= t_start) & (hours < t_end)
        h_win = hours[mask]
        v_win = values[mask]

        # Filter to finite values
        finite_mask = np.isfinite(v_win)
        h_fin = h_win[finite_mask]
        v_fin = v_win[finite_mask]

        if len(v_fin) == 0:
            # No data in this window
            for suffix in ['mean', 'std', 'range', 'auc', 'mean_rate']:
                feats[f'{prefix}_{win_name}_{suffix}'] = np.nan
            window_means[win_name] = np.nan
            continue

        # Basic statistics within the window
        feats[f'{prefix}_{win_name}_mean'] = np.mean(v_fin)
        feats[f'{prefix}_{win_name}_std'] = np.std(v_fin, ddof=1) if len(v_fin) > 1 else 0.0
        feats[f'{prefix}_{win_name}_range'] = np.ptp(v_fin)

        # AUC within the window
        if len(v_fin) > 1:
            feats[f'{prefix}_{win_name}_auc'] = np.trapz(v_fin, h_fin)
        else:
            feats[f'{prefix}_{win_name}_auc'] = 0.0

        # Mean rate of change within the window
        # (final - initial) / duration, capturing net directional change
        duration = h_fin[-1] - h_fin[0]
        if duration > 0:
            feats[f'{prefix}_{win_name}_mean_rate'] = (v_fin[-1] - v_fin[0]) / duration
        else:
            feats[f'{prefix}_{win_name}_mean_rate'] = 0.0

        window_means[win_name] = np.mean(v_fin)

    # --- Window Ratios ---
    # These capture the magnitude shift between adjacent growth phases.
    # A large mid/early ratio indicates strong activation during active growth.
    # Protected against division by zero with epsilon.
    eps = 1e-12

    early_m = window_means.get('early', np.nan)
    mid_m = window_means.get('mid', np.nan)
    late_m = window_means.get('late', np.nan)

    if np.isfinite(early_m) and abs(early_m) > eps and np.isfinite(mid_m):
        feats[f'{prefix}_mid_early_ratio'] = mid_m / early_m
    else:
        feats[f'{prefix}_mid_early_ratio'] = np.nan

    if np.isfinite(mid_m) and abs(mid_m) > eps and np.isfinite(late_m):
        feats[f'{prefix}_late_mid_ratio'] = late_m / mid_m
    else:
        feats[f'{prefix}_late_mid_ratio'] = np.nan

    return feats

print("Window features function defined (17 features per signal: 5×3 windows + 2 ratios).")

In [ ]:
# Threshold timing features
# Record the time (in hours) at which the signal first crosses
# specific fractions of its total range. This captures onset
# timing, growth rate milestones, and saturation timing without
# needing to fit a model.
#
# For monotonically increasing signals (e.g., cumulative CO2),
# this directly tracks growth milestones.
# For non-monotonic signals, the thresholds are applied to the
# running maximum to capture the time of first arrival.

def compute_threshold_timing(prefix, hours, values, thresholds):
    """
    Compute 5 threshold timing features.

    Parameters
    ----------
    prefix : str
        Feature name prefix
    hours : np.ndarray
        Time axis in hours
    values : np.ndarray
        Signal values
    thresholds : list of float
        Fractional thresholds (e.g., [0.10, 0.25, 0.50, 0.75, 0.90])

    Returns
    -------
    dict : {feature_name: value}
        Keys: t_10pct, t_25pct, t_50pct, t_75pct, t_90pct
        Values are hours at which the threshold is first crossed,
        or NaN if the signal never reaches that level.
    """
    feats = {}

    # Use finite values only
    finite_mask = np.isfinite(values)
    h_fin = hours[finite_mask]
    v_fin = values[finite_mask]

    if len(v_fin) < 2:
        for t in thresholds:
            pct_label = f'{int(t*100)}pct'
            feats[f'{prefix}_t_{pct_label}'] = np.nan
        return feats

    # Define range using running maximum (handles non-monotonic signals)
    v_min = v_fin[0]  # Initial value as baseline
    v_max = np.max(v_fin)
    total_range = v_max - v_min

    if abs(total_range) < 1e-12:
        # Signal is essentially flat — no meaningful thresholds
        for t in thresholds:
            pct_label = f'{int(t*100)}pct'
            feats[f'{prefix}_t_{pct_label}'] = np.nan
        return feats

    # Running max to handle non-monotonic signals
    running_max = np.maximum.accumulate(v_fin)

    for t in thresholds:
        pct_label = f'{int(t*100)}pct'
        target = v_min + t * total_range

        # Find first time running max exceeds the threshold
        crossings = np.where(running_max >= target)[0]
        if len(crossings) > 0:
            feats[f'{prefix}_t_{pct_label}'] = h_fin[crossings[0]]
        else:
            feats[f'{prefix}_t_{pct_label}'] = np.nan

    return feats

print("Threshold timing function defined (5 features per signal).")

## 3. Master Feature Extraction

The `extract_features_from_run()` function orchestrates all four feature groups
for all signals in a single curated data file. It returns a flat dictionary
that becomes one row of the compiled output.

In [ ]:
def extract_features_from_run(filepath, signals, windows, thresholds, smooth_window):
    """
    Extract the full FTW feature set from a single curated time-series file.

    Parameters
    ----------
    filepath : str
        Path to a curated_features_*.csv file
    signals : dict
        Signal mapping: {prefix: column_name}
    windows : dict
        Time window definitions
    thresholds : list
        Threshold fractions for timing features
    smooth_window : int
        Rolling window for rate smoothing

    Returns
    -------
    dict : All features for this run (432 values for 12 signals)
    """
    df = pd.read_csv(filepath)

    # Validate required columns
    if 'Hours' not in df.columns:
        raise ValueError(f"Missing 'Hours' column in {filepath}")

    hours = df['Hours'].values
    all_features = {}

    for prefix, col_name in signals.items():
        if col_name not in df.columns:
            # Signal column missing — fill all features with NaN
            print(f"  WARNING: Column '{col_name}' not found in {os.path.basename(filepath)}, skipping {prefix}")
            for suffix in ['mean', 'std', 'range', 'final', 'final_minus_initial',
                           'skewness', 'kurtosis', 'auc_total',
                           'max_abs_rate', 'max_rate_time_h', 'max_pos_rate',
                           'max_neg_rate', 'smooth_max_rate', 'smooth_max_rate_time_h']:
                all_features[f'{prefix}_{suffix}'] = np.nan
            for win_name in windows:
                for s in ['mean', 'std', 'range', 'auc', 'mean_rate']:
                    all_features[f'{prefix}_{win_name}_{s}'] = np.nan
            all_features[f'{prefix}_mid_early_ratio'] = np.nan
            all_features[f'{prefix}_late_mid_ratio'] = np.nan
            for t in thresholds:
                all_features[f'{prefix}_t_{int(t*100)}pct'] = np.nan
            continue

        values = df[col_name].values.astype(float)

        # --- Extract all four feature groups ---
        all_features.update(compute_global_stats(prefix, hours, values))
        all_features.update(compute_rate_features(prefix, hours, values, smooth_window))
        all_features.update(compute_window_features(prefix, hours, values, windows))
        all_features.update(compute_threshold_timing(prefix, hours, values, thresholds))

    return all_features

print(f"Master extraction function defined.")
print(f"Expected features per run: {len(SIGNALS)} signals × 36 features = {len(SIGNALS) * 36}")

## 4. Execute Pipeline

Discover all curated data files, extract features from each, and compile
into a single DataFrame.

In [ ]:
# Discover files
search_path = os.path.join(DATA_DIR, FILE_PATTERN)
files = sorted(glob.glob(search_path))

if not files:
    raise FileNotFoundError(
        f"No files matching '{FILE_PATTERN}' found in {DATA_DIR}.\n"
        f"Searched: {search_path}\n"
        f"Please check DATA_DIR and FILE_PATTERN in the configuration cell."
    )

print(f"Found {len(files)} curated data files:")
for f in files:
    print(f"  {os.path.basename(f)}")

In [ ]:
# Extract features from all files
results = {}

for filepath in files:
    fname = os.path.basename(filepath)
    print(f"Processing {fname}...", end=' ')

    try:
        feats = extract_features_from_run(
            filepath,
            signals=SIGNALS,
            windows=WINDOWS,
            thresholds=THRESHOLDS,
            smooth_window=RATE_SMOOTH_WINDOW,
        )
        results[fname] = feats
        print(f"OK ({len(feats)} features)")
    except Exception as e:
        print(f"ERROR: {e}")
        results[fname] = {}

print(f"\nExtraction complete: {len(results)} runs processed.")

In [ ]:
# Compile into DataFrame
compiled = pd.DataFrame.from_dict(results, orient='index')
compiled.index.name = 'File'

# Sort columns for consistent ordering:
# Group by signal prefix, then by feature type within each signal
def sort_key(col):
    """Sort features by signal prefix, then by feature type."""
    for i, prefix in enumerate(SIGNALS.keys()):
        if col.startswith(prefix + '_'):
            suffix = col[len(prefix)+1:]
            return (i, suffix)
    return (999, col)

compiled = compiled[sorted(compiled.columns, key=sort_key)]

# Summary statistics
n_runs, n_features = compiled.shape
n_nan = compiled.isna().sum().sum()
n_total = n_runs * n_features
nan_rate = n_nan / n_total * 100 if n_total > 0 else 0

print(f"Compiled feature matrix: {n_runs} runs × {n_features} features")
print(f"NaN count: {n_nan} / {n_total} ({nan_rate:.1f}%)")
print(f"\nFeatures per signal:")
for prefix in SIGNALS:
    n = sum(1 for c in compiled.columns if c.startswith(prefix + '_'))
    print(f"  {prefix:<12s}: {n} features")

## 5. Quality Checks

Verify the extracted features are reasonable before saving.

In [ ]:
# Check 1: expected feature count
expected = len(SIGNALS) * 36
actual = compiled.shape[1]

if actual == expected:
    print(f"PASS: Feature count matches expected ({actual} = {len(SIGNALS)} signals × 36)")
else:
    print(f"WARNING: Feature count mismatch. Expected {expected}, got {actual}")
    # Identify which signals have unexpected counts
    for prefix in SIGNALS:
        n = sum(1 for c in compiled.columns if c.startswith(prefix + '_'))
        status = 'OK' if n == 36 else f'UNEXPECTED ({n})'
        print(f"  {prefix}: {n} features — {status}")

In [ ]:
# Check 2: NaN distribution by signal
print("NaN rate by signal family:")
print("-" * 45)
for prefix in SIGNALS:
    cols = [c for c in compiled.columns if c.startswith(prefix + '_')]
    if cols:
        nan_count = compiled[cols].isna().sum().sum()
        total = len(cols) * len(compiled)
        rate = nan_count / total * 100 if total > 0 else 0
        status = 'OK' if rate < 5 else 'WARNING' if rate < 20 else 'HIGH NaN'
        print(f"  {prefix:<12s}: {rate:5.1f}% NaN  ({nan_count}/{total})  [{status}]")

In [ ]:
# Check 3: feature range sanity
# Spot-check a few features for unreasonable values
print("Feature range spot-check (first 5 runs):")
print("-" * 70)

spot_check = [
    ('cap_dt_mean', 'Capacitance mean — should be near 0 for detrended'),
    ('co2_cum_final', 'Cumulative CO2 final — should be positive and increasing'),
    ('grayscale_mid_mean', 'Mid-window grayscale — should be positive'),
    ('voc_raw_max_abs_rate', 'VOC max rate — should be non-negative'),
    ('mbi_skewness', 'MBI skewness — can be any sign'),
]

for feat, description in spot_check:
    if feat in compiled.columns:
        vals = compiled[feat].head(5).values
        print(f"  {feat}:")
        print(f"    {description}")
        print(f"    Values: {[f'{v:.4f}' for v in vals]}")
        print()

## 6. Save Compiled Features

In [ ]:
# Save to CSV
output_path = os.path.join(DATA_DIR, OUTPUT_FILENAME)
compiled.to_csv(output_path)

print(f"Features saved to: {output_path}")
print(f"  Runs:     {compiled.shape[0]}")
print(f"  Features: {compiled.shape[1]}")
print(f"  NaN rate: {nan_rate:.1f}%")
print(f"\nFile size: {os.path.getsize(output_path) / 1024:.1f} KB")

## 7. Preview

Quick look at the compiled feature matrix for visual inspection.

In [ ]:
# Show first few rows (transposed for readability since there are many columns)
print("First 3 runs, first 10 features per signal:")
print("=" * 70)

for prefix in list(SIGNALS.keys())[:4]:  # Show first 4 signals
    cols = [c for c in compiled.columns if c.startswith(prefix + '_')][:10]
    print(f"\n--- {prefix} ---")
    print(compiled[cols].head(3).round(4).to_string())

In [ ]:
# Summary statistics across all runs
print("Feature statistics across all runs:")
print("=" * 70)
desc = compiled.describe().T
desc['cv_pct'] = (desc['std'] / desc['mean'].abs() * 100).round(1)
# Show features with lowest CV (most consistent across runs)
print("\nTop 15 most consistent features (lowest CV%):")
print(desc.nsmallest(15, 'cv_pct')[['mean', 'std', 'cv_pct']].round(4).to_string())

print("\nTop 10 most variable features (highest CV%):")
print(desc.nlargest(10, 'cv_pct')[['mean', 'std', 'cv_pct']].round(4).to_string())